# DT401 Workshop 1 — SQL Part 5: Subqueries and Creating Tables

## Learning Objectives

By the end of this notebook, you will be able to:

- Write subqueries in `WHERE` and `FROM` clauses to answer multi-step questions
- Create new tables with defined column types
- Create tables populated from query results
- Remove tables using `DROP TABLE`

---

## Context Recap

In the previous notebook, you combined data from multiple tables using `JOIN`. Subqueries offer a different way to build multi-step queries — running one query inside another. This notebook also introduces `CREATE TABLE` and `DROP TABLE`, which let you modify the database structure itself rather than just read from it. Together, these tools round out the core SQL skill set covered in this workshop.

Run this cell before starting. If your kernel restarts, run it again.

In [ ]:
# Setup: load the SQL extension and connect to the pagila database
%load_ext sql
%config SqlMagic.displaylimit = 0
%config SqlMagic.autopandas = False
%config SqlMagic.autopolars = False
%config SqlMagic.displaycon = False

%sql sqlite:///data/pagila.db

---

## Section 8: Subqueries

A **subquery** (or nested query) is a query placed inside another query, enclosed in parentheses. The inner query runs first, and its result is used by the outer query — think of it as a question within a question.

Subqueries can appear in a `WHERE` clause (to filter rows based on another query's results) or in a `FROM` clause (to treat a query's output as a table that the outer query then reads from).

```sql
-- Subquery in WHERE: find customers who have made at least one payment over £10
SELECT *
FROM customer
WHERE customer_id IN (
    SELECT customer_id
    FROM payment
    WHERE amount > 10
);
```

```sql
-- Subquery in FROM: calculate the average number of transactions per customer
-- The inner query counts transactions; the outer query averages those counts
SELECT AVG(total_transactions)
FROM (
    SELECT customer_id, COUNT(*) AS total_transactions
    FROM payment
    GROUP BY customer_id
) x;
```

The alias `x` after the closing parenthesis is required when a subquery appears in a `FROM` clause — SQL needs a name for the temporary table the inner query produces.

Before working through Questions 90–92, try this warm-up to get comfortable with the `WHERE` subquery pattern.

##### Question 89a (Warm-up)

> Using a subquery in `WHERE`, find all customers who have made at least one payment of more than £10. Return their `customer_id` and `email`.

In [ ]:
%%sql


<details>
<summary>Solution to Question 89a</summary>

```sql
%%sql
SELECT customer_id, email
FROM customer
WHERE customer_id IN (
    SELECT customer_id
    FROM payment
    WHERE amount > 10
);
```

The inner query finds every `customer_id` that appears alongside a payment over £10. The outer query then retrieves the full customer details for those IDs. This is equivalent to a JOIN in many cases — but subqueries can be easier to read when the relationship is straightforward.

</details>

##### Question 90

> Using a subquery, return the average number of transactions per customer.

In [ ]:
%%sql


<details>
<summary>Solution to Question 90</summary>

```sql
%%sql
SELECT AVG(total_transactions) AS avg_transactions_per_customer
FROM (
    SELECT
        customer_id,
        COUNT(*) AS total_transactions
    FROM payment
    GROUP BY customer_id
) x;
```

The inner query counts how many payments each customer has made. The outer query then averages those counts across all customers. The result — approximately 26.8 transactions per customer — could not be obtained with a single `AVG(COUNT(*))` call; the subquery in `FROM` is the pattern that makes this kind of two-level aggregation possible.

</details>

##### Question 91

> Using a subquery, return the average number of actors featured per film.

In [ ]:
%%sql


<details>
<summary>Solution to Question 91</summary>

```sql
%%sql
SELECT AVG(num_actors) AS avg_num_actors
FROM (
    SELECT
        film_id,
        COUNT(actor_id) AS num_actors
    FROM film_actor
    GROUP BY film_id
) x;
```

The `film_actor` table records each actor–film pairing as a separate row. The inner query groups by `film_id` to count how many actors appear in each film; the outer query averages those counts. The pagila database gives approximately 5.48 actors per film on average.

</details>

##### Question 92

> Extend Question 91: using a subquery, return the average number of actors per film — broken down by film `rating`. Use `GROUP BY` in the outer query.

In [ ]:
%%sql


<details>
<summary>Solution to Question 92</summary>

```sql
%%sql
SELECT
    rating,
    AVG(num_actors) AS avg_num_actors
FROM (
    SELECT
        fa.film_id,
        f.rating,
        f.rating,
        COUNT(fa.actor_id) AS num_actors
    FROM film_actor fa
    LEFT JOIN film f ON fa.film_id = f.film_id
    GROUP BY fa.film_id, f.rating
) x
GROUP BY rating;
```

This combines the subquery pattern from Q91 with a `LEFT JOIN` inside the inner query (to pull `rating` from the `film` table) and a `GROUP BY` in the outer query. Notice that both the inner and outer queries have their own `GROUP BY` clauses — the inner groups by film, the outer groups by rating.

</details>

---

## Section 9: Creating and Removing Tables

So far you have been reading data from the pagila database. SQL also allows you to modify the database structure itself — creating new tables to store results, and removing tables you no longer need.

**Create an empty table with defined columns:**

```sql
CREATE TABLE IF NOT EXISTS my_table (
    col1 VARCHAR(250),
    col2 INT,
    col3 DATE
);
```

**Create a table populated from a query:**

```sql
CREATE TABLE IF NOT EXISTS my_table AS
SELECT *
FROM customer;
```

**Remove a table:**

```sql
DROP TABLE IF EXISTS my_table;
```

The `IF NOT EXISTS` and `IF EXISTS` clauses prevent errors when you run a cell more than once — without them, trying to create a table that already exists (or drop one that does not) will produce an error. Use them as a habit.

> ⚠️ **Warning:** `DROP TABLE` permanently deletes the table and all its data. There is no undo.

**Common column data types:**

| Type | Description |
|:-----|:------------|
| `VARCHAR(n)` | Text up to n characters |
| `CHAR(n)` | Fixed-length text of exactly n characters |
| `INT` | Integer |
| `DECIMAL(p, s)` | Decimal with p total digits and s after the decimal point |
| `TIMESTAMP` | Date and time |
| `DATE` | Date only |

All the tables in this section are named with the `x_` prefix to make them easy to identify as practice tables rather than part of the pagila database.

##### Question 93

> Create a table named `x_test` with a single `VARCHAR(250)` column called `only_column`.

In [ ]:
%%sql


<details>
<summary>Solution to Question 93</summary>

```sql
%%sql
CREATE TABLE IF NOT EXISTS x_test (only_column VARCHAR(250));
```

This creates an empty table with one text column. You can verify it exists by running `SELECT * FROM x_test;` — it will return no rows but will show the column name.

</details>

##### Question 94

> Remove the table you just created.

In [ ]:
%%sql


<details>
<summary>Solution to Question 94</summary>

```sql
%%sql
DROP TABLE IF EXISTS x_test;
```

`IF EXISTS` prevents an error if the table has already been dropped. This is particularly useful when re-running notebook cells.

</details>

##### Question 95

> Create a table named `x_test` with three columns: a text column for a product name, an integer column for quantity, and a date column for order date. Choose meaningful column names.

In [ ]:
%%sql


<details>
<summary>Solution to Question 95</summary>

```sql
%%sql
CREATE TABLE IF NOT EXISTS x_test (
    product_name VARCHAR(250),
    quantity INT,
    order_date DATE
);
```

Meaningful column names (`product_name`, `quantity`, `order_date`) make a table's purpose immediately clear to anyone reading the schema later. Generic names like `col1`, `col2`, `col3` obscure the data's meaning — a habit worth avoiding from the start.

Don't forget to drop it when you're done:

```sql
%%sql
DROP TABLE IF EXISTS x_test;
```

</details>

##### Question 96

> Create a table named `x_test` containing a full copy of the `customer` table.

In [ ]:
%%sql


In [ ]:
%%sql
DROP TABLE IF EXISTS x_test;

<details>
<summary>Solution to Question 96</summary>

```sql
%%sql
CREATE TABLE IF NOT EXISTS x_test AS
SELECT *
FROM customer;
```

`CREATE TABLE ... AS SELECT` is useful for saving the results of a complex query as a permanent table — for example, creating a summary table that other queries can reference without re-running expensive calculations each time.

</details>

##### Question 97

> Create a table named `x_test` containing only customers whose `first_name` or `last_name` begins with `M`.

In [ ]:
%%sql


In [ ]:
%%sql
DROP TABLE IF EXISTS x_test;

<details>
<summary>Solution to Question 97</summary>

```sql
%%sql
CREATE TABLE IF NOT EXISTS x_test AS
SELECT *
FROM customer
WHERE first_name LIKE 'M%'
    OR last_name LIKE 'M%';
```

The `WHERE` clause in a `CREATE TABLE ... AS SELECT` statement works exactly as it does in a regular `SELECT` — only the rows that match the condition are included in the new table.

</details>

##### Question 98

> Create a table named `x_test` with these specific column types: `DECIMAL(8,2)`, `INT`, `VARCHAR(10)`, `CHAR(10)`, `TIMESTAMP`. Give each column a meaningful name related to a sales transaction.

In [ ]:
%%sql


In [ ]:
%%sql
DROP TABLE IF EXISTS x_test;

<details>
<summary>Solution to Question 98</summary>

```sql
%%sql
CREATE TABLE IF NOT EXISTS x_test (
    sale_amount DECIMAL(8, 2),
    quantity INT,
    product_code VARCHAR(10),
    store_code CHAR(10),
    sale_timestamp TIMESTAMP
);
```

`DECIMAL(8, 2)` allows up to 8 digits in total, with 2 after the decimal point — suitable for a monetary amount up to £999,999.99. `CHAR(10)` is appropriate for fixed-length codes (such as store identifiers) where every value is always exactly 10 characters; `VARCHAR(10)` is better for codes that vary in length.

</details>

##### Question 99

> Create a table named `x_test` containing customers with first names in the list `('MOLLY', 'JENNY', 'JIMMY', 'LENNY')`.

In [ ]:
%%sql


In [ ]:
%%sql
DROP TABLE IF EXISTS x_test;

<details>
<summary>Solution to Question 99</summary>

```sql
%%sql
CREATE TABLE IF NOT EXISTS x_test AS
SELECT *
FROM customer
WHERE first_name IN ('MOLLY', 'JENNY', 'JIMMY', 'LENNY');
```

This combines `CREATE TABLE ... AS SELECT` with the `IN` operator from Notebook 2. The new table will contain only the rows matching those first names — a useful pattern when you need to extract a specific subset of records into a separate table for further analysis.

</details>

##### Question 100

> Create a table named `x_test` based on the `actor` table, containing only the `first_name` and `last_name` columns.

In [ ]:
%%sql


In [ ]:
%%sql
DROP TABLE IF EXISTS x_test;

<details>
<summary>Solution to Question 100</summary>

```sql
%%sql
CREATE TABLE IF NOT EXISTS x_test AS
SELECT first_name, last_name
FROM actor;
```

Selecting specific columns rather than `SELECT *` is good practice when creating a derived table — it keeps the table focused on what is actually needed and avoids carrying over columns that may cause confusion or consume unnecessary storage.

</details>

---

## Summary

In this notebook you have covered:

| Keyword / Function | Purpose |
|:-------------------|:--------|
| Subquery in `WHERE` | Filter rows using results from an inner query |
| Subquery in `FROM` | Use a query's output as a temporary table for the outer query |
| `CREATE TABLE IF NOT EXISTS name (...)` | Create an empty table with defined columns |
| `CREATE TABLE IF NOT EXISTS name AS SELECT ...` | Create a table populated from a query result |
| `DROP TABLE IF EXISTS name` | Permanently remove a table |

**Next:** [6 — Advanced Practice](6_advanced_practice.ipynb)